# Tool Request

ToolCallRequest  - Level 1 — ToolCallRequest 
│
├── tool_call    - Level 2 — request.tool_call - This is the actual request from the LLM: tool_call → request/order
│     ├── name
│     ├── args
│     ├── id
│     └── type
│
├── tool          - Level 3 — request.tool - This is the actual tool object: (StructuredTool(name='read_file', ...)) . tool → actual worker
│     └── StructuredTool
│
├── state - Level 4 — request.state - "What is the agent's current state when this tool is being called?"
│     ├── messages
│     ├── structured_response
│     └── memory_contents
│
└── runtime - Level 5 — request.runtime - This is the execution environment provided to the tool: 
      │
      └── ToolRuntime 
            │
            ├── state - It isn't a second independent state.. It's the state made available through the tool's runtime context
            ├── context
            ├── config
            │     │
            │     └── configurable
            │           │
            │           └── __pregel_runtime
            │                 │
            │                 └── Runtime - This Runtime is LangGraph's internal runtime. So don't confuse: ToolRuntime with Runtime.They are diff
            │                       ├── context
            │                       ├── store
            │                       ├── stream_writer
            │                       ├── execution_info
            │                       └── control
            │
            ├── stream_writer
            ├── tool_call_id
            ├── store
            ├── tools
            └── execution_info

# Level - 1 - ToolCallRequest : 
    Level 1 — ToolCallRequest - This is what your wrap_tool_call() receives: def wrap_tool_call(self, request, handler): 
    It is basically: "A tool is about to be executed. Here is everything you need to know."

# Level - 2 - request.tool_call - WHAT does the agent want?
    This is the actual request from the LLM:
     request.tool_call["name"]
     request.tool_call["args"]

# Level 3 — request.tool - This is the actual tool object:
    StructuredTool(name='read_file', ...)

# Level 4 — request.state - This is the current Deep Agent state.

# Level 5 — request.runtime - This is the execution environment provided to the tool:



# Tool Response

response
│
├── ToolMessage
│     ├── name
│     ├── content
│     ├── tool_call_id
│     └── status
│
OR
│
└── Command
      └── update / graph-control information

# Model Request

wrap_model_call()
│
├── request: ModelRequest
│   │
│   ├── model
│   │   └── ChatModel / LLM object
│   │
│   ├── system_message
│   │   └── SystemMessage
│   │       ├── content
│   │       ├── additional_kwargs
│   │       ├── response_metadata
│   │       ├── id
│   │       └── ...
│   │
│   ├── messages
│   │   └── list[BaseMessage]
│   │       │
│   │       ├── HumanMessage
│   │       │   ├── content
│   │       │   ├── additional_kwargs
│   │       │   ├── response_metadata
│   │       │   ├── id
│   │       │   └── ...
│   │       │
│   │       ├── AIMessage
│   │       │   ├── content
│   │       │   ├── tool_calls
│   │       │   ├── usage_metadata
│   │       │   ├── additional_kwargs
│   │       │   ├── response_metadata
│   │       │   ├── id
│   │       │   └── ...
│   │       │
│   │       └── ToolMessage
│   │           ├── content
│   │           ├── name
│   │           ├── tool_call_id
│   │           ├── status
│   │           ├── additional_kwargs
│   │           ├── response_metadata
│   │           ├── id
│   │           └── ...
│   │
│   ├── tools
│   │   └── list of tools available to the model
│   │       ├── tool name
│   │       ├── description
│   │       ├── args/schema
│   │       └── ...
│   │
│   ├── state
│   │   └── current agent state
│   │       ├── messages
│   │       ├── ...
│   │       └── custom state fields
│   │
│   ├── runtime
│   │   │
│   │   ├── context
│   │   │   └── your application context
│   │   │       └── user_id
│   │   │
│   │   ├── store
│   │   │   └── LangGraph Store
│   │   │
│   │   ├── stream_writer
│   │   │
│   │   ├── execution_info
│   │   │   ├── ...
│   │   │   └── ...
│   │   │
│   │   └── ...
│   │
│   ├── model_settings
│   │   ├── temperature
│   │   ├── max_tokens
│   │   ├── ...
│   │   └── ...
│   │
│   ├── request_id
│   │   └── ...
│   │
│   └── ...
│
└── handler(request)
    │
    ▼
    Model
    │
    │  model receives the request
    │
    ▼
    ModelResponse
    │
    ├── result
    │   └── AIMessage
    │       ├── content
    │       ├── tool_calls
    │       │   └── [
    │       │       {
    │       │         name
    │       │         args
    │       │         id
    │       │         type
    │       │       }
    │       │     ]
    │       │
    │       ├── usage_metadata
    │       │   ├── input_tokens
    │       │   ├── output_tokens
    │       │   ├── total_tokens
    │       │   ├── input_token_details
    │       │   └── output_token_details
    │       │
    │       ├── response_metadata
    │       │   ├── model_name
    │       │   ├── finish_reason
    │       │   └── ...
    │       │
    │       ├── additional_kwargs
    │       ├── id
    │       └── ...
    │
    └── ...

# Model Response

request
   │
   ├── messages ───────────────► What the LLM is being given
   ├── system_message ────────► System instructions
   ├── tools ──────────────────► Tools the LLM can choose
   ├── state ──────────────────► Current agent state
   ├── runtime ────────────────► Execution environment
   ├── model ──────────────────► Actual LLM
   └── model settings ─────────► Model configuration
                │
                ▼
         handler(request)
                │
                ▼
             LLM call
                │
                ▼
           ModelResponse
                │
                └── result
                     │
                     ▼
                  AIMessage
                     ├── content
                     ├── tool_calls
                     ├── usage_metadata
                     ├── response_metadata
                     └── ...

# Storage Architecture

                         AEP
                          │
             ┌────────────┴────────────┐
             │                         │
       LangGraph State            Agent Outputs
       / Checkpoints              / Artifacts
             │                         │
             ▼                         ▼
        PostgreSQL                Object Storage
       (durable state)          (S3 / Azure Blob / GCS)
             │
             │
             ▼
       Persistent Store
       (memory / learning /
        metadata / refs)
             │
             ▼
        PostgreSQL

# 1. Persistent Store → PostgreSQL

1. Persistent Store → PostgreSQL

Replace your current: InMemoryStore() with a PostgreSQL-backed store.

Use PostgreSQL for things like:

LEARNINGS.md / learned rules
AGENTS.md
personal memory
small metadata
artifact metadata/references
user/agent scoped information

This is the data you need to survive process restarts.

# 2. Agent Output → Object Storage

I agree with your idea of using Blob/Object Storage for agent outputs.

For example:

s3://aep-artifacts/
    users/
        user123/
            executions/
                execution456/
                    repository_analysis.json
                    report.md
                    logs.json

or Azure Blob:

aep-artifacts/
    user123/
        execution456/
            repository_analysis.json
            report.md

Your current:

artifact_id = str(uuid4())
self.store.put(...)

would become something conceptually like:

agent produces output
        ↓
serialize output
        ↓
Object Storage
        ↓
artifact_id
        ↓
PostgreSQL metadata
        ↓
CustomState.artifacts_id

So CustomState still only carries:

artifacts_id

rather than carrying the actual large output.

I strongly recommend keeping this separation.

# 3. Checkpointer → PostgreSQL

This is separate from the Store.

Your current:

self.checkpointer = MemorySaver()

should not remain MemorySaver in production.

Use a PostgreSQL-backed LangGraph checkpointer.

The purpose is different:

Checkpointer
    ↓
"Where was this graph execution?"
"What was its state?"
"Can I resume this thread?"

Whereas your Store is:

Store
    ↓
"What persistent information does the agent know?"

And Artifact Storage is:

Artifact Storage
    ↓
"What did the agent produce?"

That's three different responsibilities.

# 4. Where should CustomState live?



You don't need to separately create a CustomState database.

This is the important part.

When the graph executes:

CustomState

LangGraph's checkpointer persists the graph state/checkpoints.

So conceptually:

Outer Graph
     │
     │ CustomState
     ▼
PostgreSQL Checkpointer
     │
     ├── thread_id
     ├── checkpoint
     ├── state
     └── execution history

You should therefore let the LangGraph checkpointer own execution state rather than building another state database.

# So I would use this production architecture

| Data                              | Production storage          | Purpose                            |
| --------------------------------- | --------------------------- | ---------------------------------- |
| `CustomState` / graph checkpoints | **PostgreSQL Checkpointer** | Resume/recover graph execution     |
| Agent memory                      | **PostgreSQL Store**        | Persistent memory                  |
| `LEARNINGS.md` equivalent         | **PostgreSQL Store**        | Reusable lessons                   |
| Personal memory                   | **PostgreSQL Store**        | User-specific memory               |
| Agent output                      | **Blob/Object Storage**     | Large detailed outputs             |
| Artifact metadata                 | **PostgreSQL**              | ID → location/type/owner/execution |
| Sandbox                           | **Temporary sandbox**       | Working area during execution      |


                  ┌───────────────┐
                  │   AEP Graph   │
                  └───────┬───────┘
                          │
                    CustomState
                          │
             ┌────────────┴────────────┐
             │                         │
             ▼                         ▼
     LangGraph Checkpointer       Agent execution
        PostgreSQL                     │
                                       │
                              ┌────────┴────────┐
                              │                 │
                              ▼                 ▼
                       Persistent Store     Agent Output
                        PostgreSQL         Object Storage
                              │                 │
                              │                 ▼
                              │          artifact_id
                              │                 │
                              └────────┬────────┘
                                       ▼
                              AEP context layer

One more thing I would change

Don't store the entire artifact location only in CustomState.

Keep:

artifacts_id

in CustomState, as you have now.

But maintain an artifact metadata record in PostgreSQL:

artifact_id
user_id
execution_id
agent_name
artifact_type
object_storage_key
created_at
content_type

Then:

CustomState
    │
    │ artifact_id
    ▼
PostgreSQL metadata
    │
    │ object_storage_key
    ▼
Blob Storage
    │
    ▼
actual agent output

This gives you a clean architecture where PostgreSQL is the control plane and Blob Storage is the data plane.


Step 1: Replace InMemoryStore with persistent PostgreSQL Store.

Step 2: Replace MemorySaver with PostgreSQL Checkpointer.

Step 3: Move store_agent_output() / get_agent_output() to Blob Storage + PostgreSQL artifact metadata.

Step 4: Update ContextManager to retrieve artifacts through the new artifact service.

Step 5: Review the whole persistence architecture for production concerns such as connection pooling, transactions, TTL/cleanup, tenant isolation, encryption, and observability.

I would start with Step 1 only. Your existing backend class is already nicely isolated, so this should be a relatively contained change.

| Priority | Tweak                                        | Why                                  |
| -------- | -------------------------------------------- | ------------------------------------ |
| **1**    | ContextManager filters context               | Biggest current problem  - Done             |
| **2**    | Separate execution result vs verification    | Prevents false validation - Done           |
| **3**    | Improve execution-memory structure           | Prevents losing pending work - DOne       |
| **4**    | Explicit memory retrieval/injection          | Proves memory is actually being used - Done |
| **5**    | Push status (`success/failed/not_attempted`) | Makes execution state reliable - Dome    |
| **6**    | Secure credential handling                   | Needed for real GitHub push          |
| **7**    | Further learning improvements                | Polish after core flow works         |


# MCP

Step 1 — In-process MCP - done

Understand the smallest possible MCP setup.
Create an MCP server inside our AEP process.
Expose one simple tool.
Connect to it through MCPAdapter.
Make the Deep Agent see it as a normal LangChain tool.

Step 2 — stdio MCP  - done

Move the MCP server into a separate process.
AEP communicates with it through stdin/stdout.
Understand why this is useful.

Step 3 — Streamable HTTP MCP - - done

Run the MCP server separately as a remote service.
Connect using an HTTP MCP endpoint.
This is much closer to our production architecture.

Step 4 — MCP authentication - - done

Understand bearer-token authentication.
Then OAuth 2.1.
Understand where credentials should live.
Make sure credentials never enter prompts, state, memory, artifacts, or logs.

Step 5 — GitHub MCP - done

Connect AEP → MCP → GitHub.
Give the agent GitHub operations through MCP rather than directly implementing GitHub API calls in AEP.

Step 6 — Human-in-the-loop  - done

Identify destructive GitHub operations.
Pause before those operations.
Human approves/rejects.
Resume using LangGraph's interrupt/checkpointer mechanism.

Step 7 — Per-user authentication - onhold

Move from one credential to user-specific credentials.
Understand how the AEP identifies the user and obtains the appropriate GitHub token.

Step 8 — Production connection management

Connection pooling.
Tool discovery/caching.
Multiple MCP servers.
ClientGroup vs MCPConfig.
Protocol-version considerations.

Then, only after this first MCP document is properly implemented, we'll move to the second document: Document Connectors.

Agent decides → "call subtraction"
        ↓
LangChain tool invocation
        ↓
MCPAdapter
        ↓
MCP protocol call
        ↓
FastMCP server's subtraction()
        ↓
result comes back
        ↓
ToolMessage
        ↓
Agent continues reasoning

|                          | Bearer Token                                | OAuth                                                                |
| ------------------------ | ------------------------------------------- | -------------------------------------------------------------------- |
| What is it?              | A credential/token                          | An authorization framework                                           |
| Main purpose             | Prove possession of a valid token           | Obtain/manage delegated access                                       |
| Who issues it?           | Could be manually generated by server/admin | Authorization Server                                                 |
| User authentication flow | Not necessarily                             | Yes, typically                                                       |
| Scopes/permissions       | Can exist, but not inherent                 | Core part of OAuth authorization                                     |
| Token lifecycle          | You manage it                               | OAuth defines mechanisms around obtaining/refreshing/revoking access |
| Per-user access          | Possible, but you'd have to build it        | Natural fit                                                          |
| Example                  | `my-secret-token`                           | "Sign in with GitHub → authorize → receive token"                    |


|                | MCPConfig                 | ClientGroup                    |
| -------------- | ------------------------- | ------------------------------ |
| Main purpose   | Describe connections      | Manage clients                 |
| Think of it as | Configuration             | Runtime manager                |
| Contains       | Server connection details | Client instances/connections   |
| Concern        | *What should connect?*    | *How are connections managed?* |


                         AEP
                          │
                   Connection Manager
                          │
                     MCPConfig
                          │
                     ClientGroup
                          │
          ┌───────────────┼───────────────┐
          ↓               ↓               ↓
       GitHub            Jira          Internal
        MCP               MCP             MCP
          │               │               │
          ↓               ↓               ↓
      MCP Session      MCP Session     MCP Session
          │               │               │
          └───────────────┼───────────────┘
                          ↓
                Connection Pooling
                          +
                 Tool Discovery
                          +
                    Tool Cache
                          +
               Protocol Compatibility

We covered:

✅ Connection pooling
✅ Tool discovery
✅ Tool-definition caching
✅ Cache invalidation/refresh concept
✅ Multiple MCP servers
✅ MCPConfig vs ClientGroup
✅ MCP protocol versions
✅ Connection Manager concept

# Cache Managememt


1. Time-based refresh (TTL)

For example:

Discover → cache for 1 hour
After 1 hour → discover again

2. Connection/session lifecycle

When establishing a new MCP connection/session, perform discovery again.

3. Explicit refresh

If an administrator changes an MCP server or deploys a new version:

Refresh GitHub tools

4. Server/version change

If we can detect that the server/version/capabilities changed, invalidate the cache.

Our implementation order

We'll do it in this order:

Step 1 — MCPManager boundary  - Done
→ Connect your existing MCPAdapter/MCP server to AEP.

Step 2 — Multiple MCP servers - Done
→ MCPManager handles several servers.

Step 3 — Tool discovery + caching - Done
→ Don't rediscover tools unnecessarily.

Step 4 — Connection management + pooling - DOne
        4.1 - 4D Pooling/lifecycle ✅
        4.2 - 4E Broken-session handling
→ Reuse MCP connections appropriately.

Step 5 — MCPConfig vs ClientGroup - Done
→ Decide where these fit and whether we actually need them.

Step 6 — Protocol versions - DOne
→ Handle MCP client/server version compatibility.

Step 7 — Structured results - Done
→ Connect MCP artifact/structured_content to your AEP artifact architecture.
  
Step 8 — MCP errors  - Done
→ Separate tool errors from transport/session failures.

Step 9 — Destructive annotations + HITL  - Done
→ Automatically put approval gates around destructive MCP tools.

Step 10 — Elicitation - Done
→ Integrate server-requested human input.

And later, when we revisit tool discovery/cache invalidation, we'll properly handle:

refresh=True
updating the cache
updating pooled entries
concurrent agents using old tools
server capability/tool changes

AEP
│
└── MCPManager
    │
    ├── Server: github_1
    │   │
    │   └── Client
    │       │
    │       └── Session Pool
    │           ├── Session 1
    │           ├── Session 2
    │           ├── Session 3
    │           ├── Session 4
    │           └── Session 5
    │
    └── Server: github_2
        │
        └── Client
            │
            └── Session Pool
                ├── Session 1
                ├── Session 2
                ├── Session 3
                ├── Session 4
                └── Session 5

OUTER GRAPH
AgentExecutor.graph
│
├── details_gathering_node
│
├── outer_graph_state_tracking_before
│
├── deep_agent_executor          ← NODE in your outer graph
│       │
│       │ creates
│       ▼
│   create_deep_agent(...)
│       │
│       ▼
│   deep_agent.ainvoke(...)
│       │
│       │ HITL middleware
│       ▼
│   interrupt()
│
├── outer_graph_state_tracking_after
│
├── store_tracking
│
└── end_node

In [ ]:
So, in your terminology:

Outer graph: self.graph / AgentExecutor.graph
Outer graph node: deep_agent_executor
Inner graph: the graph created internally by create_deep_agent(...)
No manually defined child node: correct
HITL: middleware inside that Deep Agent
Interrupt: generated during the inner Deep Agent's execution

             AEP MEMORY MODEL

       ┌─────────────────────────┐
       │   Semantic Memory       │
       │                         │
       │  "What do we know?"     │
       │                         │
       │ AGENTS.md               │
       │ LEARNINGS.md            │
       │ USER_MEMORY.md          │
       └─────────────────────────┘

       ┌─────────────────────────┐
       │   Episodic Memory       │
       │                         │
       │  "What happened?"       │
       │                         │
       │ Checkpointed threads    │
       │ Conversation history    │
       │ Execution history       │
       └─────────────────────────┘

       ┌─────────────────────────┐
       │   Working State         │
       │                         │
       │  "Where are we now?"    │
       │                         │
       │ Current graph state     │
       │ Pending interrupt       │
       │ Current task            │
       └─────────────────────────┘

In [ ]:
Use durable thread_ids.

New Chat → generate new thread_id
Continue Chat → reuse existing thread_id
Browser close/reopen → don't automatically create a new thread
Checkpointer → persists conversation/execution state for that thread
Store → durable memory that may span multiple threads
No custom execution_id needed yet

That's the architecture I'd use for your AEP at this stage.

In [ ]:
What does build_context() do?
        │
        ├── normal synchronous logic
        │       → keep def
        │
        ├── async I/O
        │       → async def + await
        │       → gather if contexts are independent
        │
        ├── blocking synchronous I/O
        │       → consider asyncio.to_thread()
        │
        └── heavy CPU work
                → consider process-based execution